In [1]:
# ==============================
# CELL 1 — GPU SETUP
# ==============================

import torch
import os

assert torch.cuda.is_available(), "CUDA is required. No GPU detected."

device = torch.device("cuda")
print("Using GPU:", torch.cuda.get_device_name(0))
print("Total Memory (GB):", torch.cuda.get_device_properties(0).total_memory / 1e9)

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True

SAVE_DIR = "./ControlNetModels2"
os.makedirs(SAVE_DIR, exist_ok=True)

Using GPU: NVIDIA H100 80GB HBM3
Total Memory (GB): 85.017624576


In [2]:
# ==============================
# CELL 2 — IMPORTS
# ==============================

import json
import random
import numpy as np
from glob import glob
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as T
from torchvision.models import inception_v3

from diffusers import (
    AutoencoderKL,
    UNet2DConditionModel,
    ControlNetModel,
    DDPMScheduler
)

from transformers import CLIPTokenizer, CLIPTextModel
from torch.amp import GradScaler, autocast

/home/haree/scratch/haree/venv/xraysynthetic/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ==============================
# CELL 3 — PATHS & DATA LOADING
# ==============================

DATA_ROOT = os.path.expanduser("~/scratch/haree/data/det_compass_complete")

IMG_DIR = os.path.join(DATA_ROOT, "images")
ANNOTATION_FILE = os.path.join(DATA_ROOT,"annotations.json")
COLOUR_FILE = os.path.join(DATA_ROOT,"colour_per_cat.json")
MATERIAL_FILE = os.path.join(DATA_ROOT,"material_classification.json")

with open(ANNOTATION_FILE) as f:
    annotations = json.load(f)

with open(COLOUR_FILE) as f:
    colour_map = json.load(f)

with open(MATERIAL_FILE) as f:
    material_map = json.load(f)

# Reverse material mapping
category_to_material = {}
for material, categories in material_map.items():
    for c in categories:
        category_to_material[c] = material

print("Loaded annotations:", len(annotations))

Loaded annotations: 1573


In [4]:
# ==============================
# CELL 4 — UTILITY FUNCTIONS
# ==============================

def load_image(path, size=(512,512), grayscale=False):
    img = Image.open(path).convert("L" if grayscale else "RGB")
    tf = T.Compose([
        T.Resize(size),
        T.ToTensor()
    ])
    return tf(img)

def make_bbox_mask(bbox, img_size=(512,512)):
    mask = torch.zeros(1, *img_size)
    if bbox is None:
        return mask
    
    x,y,w,h = bbox
    x1 = int(x)
    y1 = int(y)
    x2 = int(x + w)
    y2 = int(y + h)
    mask[:, y1:y2, x1:x2] = 1.0
    return mask

def compute_edge_map(img):
    sobel_x = torch.tensor([[1,0,-1],[2,0,-2],[1,0,-1]], dtype=torch.float32, device=img.device).unsqueeze(0).unsqueeze(0)
    sobel_y = sobel_x.transpose(-1,-2)
    gray = img.mean(dim=1, keepdim=True)
    gx = F.conv2d(gray, sobel_x, padding=1)
    gy = F.conv2d(gray, sobel_y, padding=1)
    edges = torch.sqrt(gx**2 + gy**2)
    return edges.repeat(1,3,1,1)

In [5]:
# ==============================
# CELL 5 — DATASET CLASS
# ==============================

class XrayDataset(Dataset):
    def __init__(self, annotations):
        self.samples = []
        
        for ann in annotations:
            try:
                rgb_path = os.path.join(IMG_DIR, ann["rgb_file"])
                xray_paths = [os.path.join(IMG_DIR, f) for f in ann["xray_files"]]
                
                if not all(os.path.exists(p) for p in [rgb_path] + xray_paths):
                    continue
                
                category = ann["objects"]["categories"][0]
                bbox = ann["objects"]["xray_bbox"][0]
                
                material = category_to_material.get(category, "unknown")
                colour = colour_map.get(category, [128,128,128])
                
                self.samples.append({
                    "rgb": rgb_path,
                    "xray": xray_paths,
                    "category": category,
                    "material": material,
                    "colour": colour,
                    "bbox": bbox
                })
            except:
                continue
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        s = self.samples[idx]
        
        rgb = load_image(s["rgb"])
        xray_col = load_image(s["xray"][0])
        xray_gray = load_image(s["xray"][1], grayscale=True)
        xray_high = load_image(s["xray"][2], grayscale=True)
        xray_low = load_image(s["xray"][3], grayscale=True)
        xray_density = load_image(s["xray"][4], grayscale=True)
        
        bbox_mask = make_bbox_mask(s["bbox"])
        
        return {
            "rgb": rgb,
            "xray_col": xray_col,
            "gray": xray_gray,
            "high": xray_high,
            "low": xray_low,
            "density": xray_density,
            "bbox_mask": bbox_mask,
            "category": s["category"],
            "material": s["material"],
            "colour": torch.tensor(s["colour"], dtype=torch.float32) / 255.0
        }

In [6]:
# ==============================
# CELL 6 — SPLIT
# ==============================

dataset = XrayDataset(annotations)

n = len(dataset)
train_n = int(0.7 * n)
val_n = int(0.15 * n)
test_n = n - train_n - val_n

train_set, val_set, test_set = torch.utils.data.random_split(
    dataset, [train_n, val_n, test_n]
)

train_loader = DataLoader(train_set, batch_size=4, shuffle=True, num_workers=8, pin_memory=True)
val_loader = DataLoader(val_set, batch_size=4, shuffle=False, num_workers=8, pin_memory=True)

print("Train:", len(train_set), "Val:", len(val_set), "Test:", len(test_set))

Train: 1082 Val: 231 Test: 233


In [7]:
# ==============================
# CELL 7 — MODEL (FULL MULTIMODAL)
# ==============================

model_id = "runwayml/stable-diffusion-v1-5"

vae = AutoencoderKL.from_pretrained(model_id, subfolder="vae").to(device)
unet = UNet2DConditionModel.from_pretrained(model_id, subfolder="unet").to(device)
text_encoder = CLIPTextModel.from_pretrained(model_id, subfolder="text_encoder").to(device)
tokenizer = CLIPTokenizer.from_pretrained(model_id, subfolder="tokenizer")

# 8 spatial conditioning channels:
# edges(3) + bbox(1) + gray(1) + high(1) + low(1) + density(1)
CONTROL_CHANNELS = 8

controlnet = ControlNetModel.from_unet(
    unet,
    conditioning_channels=CONTROL_CHANNELS
).to(device)

# Freeze backbone
for p in vae.parameters():
    p.requires_grad = False

for p in unet.parameters():
    p.requires_grad = False

for p in text_encoder.parameters():
    p.requires_grad = False

scheduler = DDPMScheduler.from_pretrained(model_id, subfolder="scheduler")

print("ControlNet conditioning channels:", CONTROL_CHANNELS)

/home/haree/scratch/haree/venv/xraysynthetic/lib/python3.11/site-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████| 196/196 [00:00<00:00, 1215.37it/s, Materializing param=text_model.final_layer_norm.weight]
CLIPTextModel LOAD REPORT from: runwayml/stable-diffusion-v1-5
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ControlNet conditioning channels: 8


In [8]:
# ==============================
# CELL 8 — METADATA FUSION
# ==============================

class MetadataFusion(nn.Module):
    def __init__(self):
        super().__init__()
        self.colour_mlp = nn.Sequential(
            nn.Linear(3, 128),
            nn.ReLU(),
            nn.Linear(128, 768)
        )
        self.fuse = nn.Linear(768 * 2, 768)

    def forward(self, cls_token, colour):
        colour_emb = self.colour_mlp(colour)
        fused = self.fuse(torch.cat([cls_token, colour_emb], dim=-1))
        return fused

fusion_module = MetadataFusion().to(device)

params = list(controlnet.parameters()) + list(fusion_module.parameters())
optimizer = torch.optim.AdamW(params, lr=1e-5)

scaler = GradScaler("cuda")

In [31]:
# ==============================
# CELL 9 — LOSSES & METRICS
# ==============================

import numpy as np
from scipy import linalg
from torchvision.models import inception_v3, Inception_V3_Weights

# ----------------------------
# Loss Weights
# ----------------------------
LAMBDA_DIFF = 1.0
LAMBDA_RECON = 1.0
LAMBDA_AUX = 0.5
LAMBDA_BBOX = 0.5

# ----------------------------
# Diffusion Loss
# ----------------------------
def diffusion_loss(pred, target):
    return F.mse_loss(pred, target)

def reconstruction_loss(pred, target):
    return F.l1_loss(pred, target)


# ------------------------------------------------
# InceptionV3 Feature Extractor
# ------------------------------------------------

weights = Inception_V3_Weights.IMAGENET1K_V1

inception = inception_v3(
    weights=weights,
    transform_input=False,
)

# Remove classifier
inception.fc = nn.Identity()

inception = inception.to(device)
inception.eval()


# ------------------------------------------------
# Feature Extraction
# ------------------------------------------------

@torch.no_grad()
def get_inception_features(images):

    if images.shape[-1] != 299:
        images = F.interpolate(images, size=(299,299), mode="bilinear")

    images = (images + 1) / 2
    images = images.clamp(0,1)

    feats = inception(images)

    return feats


# ------------------------------------------------
# FID
# ------------------------------------------------

def compute_fid(real_feats, fake_feats):

    real_feats = real_feats.cpu().numpy()
    fake_feats = fake_feats.cpu().numpy()

    mu1 = np.mean(real_feats, axis=0)
    mu2 = np.mean(fake_feats, axis=0)

    sigma1 = np.cov(real_feats, rowvar=False)
    sigma2 = np.cov(fake_feats, rowvar=False)

    diff = mu1 - mu2

    covmean = linalg.sqrtm(sigma1 @ sigma2)

    if np.iscomplexobj(covmean):
        covmean = covmean.real

    fid = diff @ diff + np.trace(sigma1 + sigma2 - 2 * covmean)

    return float(fid)


# ------------------------------------------------
# FAST SID (Kernel Density Difference)
# ------------------------------------------------
def compute_sid_X(real_feats, fake_feats, num_test_points=512, sigma=10.0):

    device = real_feats.device

    real_feats = real_feats.detach()
    fake_feats = fake_feats.detach()

    N = real_feats.shape[0]
    D = real_feats.shape[1]

    # ---------------------------------
    # Sample test points in hypercube
    # ---------------------------------

    combined = torch.cat([real_feats, fake_feats], dim=0)

    feat_min = combined.min(dim=0)[0]
    feat_max = combined.max(dim=0)[0]

    rand = torch.rand(num_test_points, D, device=device)

    test_points = feat_min + rand * (feat_max - feat_min)

    # ---------------------------------
    # Gaussian kernel
    # ---------------------------------

    def rbf(x, y):
        dist = torch.cdist(x, y) ** 2
        return torch.exp(-dist / (2 * sigma**2))

    k_fake = rbf(test_points, fake_feats)
    k_real = rbf(test_points, real_feats)

    sid = (k_fake.mean(dim=1) - k_real.mean(dim=1)).mean()

    return sid.item()
    
def compute_sid(real_feats, fake_feats):

    real_feats = real_feats.detach()
    fake_feats = fake_feats.detach()

    # feature statistics
    real_var = real_feats.var(dim=0).mean()
    fake_var = fake_feats.var(dim=0).mean()

    # diversity difference
    diversity_gap = real_var - fake_var

    # realism difference
    fid = compute_fid(real_feats, fake_feats)

    sid = diversity_gap * torch.sqrt(torch.tensor(fid))

    return sid.item()

# ------------------------------------------------
# Metric Wrapper
# ------------------------------------------------

@torch.no_grad()
def evaluate_metrics(real_images, fake_images):

    real_feats = get_inception_features(real_images)
    fake_feats = get_inception_features(fake_images)

    fid = compute_fid(real_feats, fake_feats)

    sid = compute_sid(real_feats, fake_feats)

    return fid, sid

In [ ]:
# ==============================
# CELL 10 — TRAIN + VALIDATION 
# ==============================

EPOCHS = 100
best_val_loss = float("inf")

alphas_cumprod = scheduler.alphas_cumprod.to(device)


# ==============================
# TRAINING LOOP
# ==============================

for epoch in range(EPOCHS):

    # -------------------------
    # TRAIN
    # -------------------------
    controlnet.train()
    fusion_module.train()

    train_loss_total = 0

    for batch in tqdm(train_loader):

        optimizer.zero_grad(set_to_none=True)

        rgb = batch["rgb"].to(device)
        target = batch["xray_col"].to(device)

        gray = batch["gray"].to(device)
        high = batch["high"].to(device)
        low = batch["low"].to(device)
        density = batch["density"].to(device)

        bbox_mask = batch["bbox_mask"].to(device)
        colour = batch["colour"].to(device)

        # Encode GT X-ray
        with torch.no_grad():
            latents = vae.encode(target).latent_dist.sample() * 0.18215

        noise = torch.randn_like(latents)

        timesteps = torch.randint(
            0,
            scheduler.config.num_train_timesteps,
            (latents.size(0),),
            device=device
        ).long()

        noisy_latents = scheduler.add_noise(latents, noise, timesteps)

        # Spatial conditioning
        edges = compute_edge_map(rgb)

        control_input = torch.cat([
            edges,
            bbox_mask,
            gray,
            high,
            low,
            density
        ], dim=1)

        # Metadata prompt
        prompts = [
            f"airport x-ray image of a {m} {c}"
            for m, c in zip(batch["material"], batch["category"])
        ]

        text_inputs = tokenizer(
            prompts,
            padding="max_length",
            truncation=True,
            max_length=77,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            text_emb = text_encoder(**text_inputs).last_hidden_state

        cls_token = text_emb[:,0,:]
        fused_cls = fusion_module(cls_token, colour)
        text_emb[:,0,:] = fused_cls

        encoder_hidden_states = text_emb

        # Forward
        with autocast("cuda"):

            down_samples, mid_sample = controlnet(
                noisy_latents,
                timesteps,
                encoder_hidden_states=encoder_hidden_states,
                controlnet_cond=control_input,
                return_dict=False
            )

            noise_pred = unet(
                noisy_latents,
                timesteps,
                encoder_hidden_states=encoder_hidden_states,
                down_block_additional_residuals=down_samples,
                mid_block_additional_residual=mid_sample
            ).sample

            loss_diff = diffusion_loss(noise_pred, noise)

            alpha_t = alphas_cumprod[timesteps].view(-1,1,1,1)

            sqrt_alpha = torch.sqrt(alpha_t)
            sqrt_one_minus = torch.sqrt(1 - alpha_t)

            pred_x0 = (noisy_latents - sqrt_one_minus * noise_pred) / sqrt_alpha

            recon = vae.decode(pred_x0 / 0.18215).sample

            loss_recon = reconstruction_loss(recon, target)

            recon_gray = recon.mean(1, keepdim=True)

            loss_gray = reconstruction_loss(recon_gray, gray)
            loss_high = reconstruction_loss(recon_gray, high)
            loss_low = reconstruction_loss(recon_gray, low)
            loss_density = reconstruction_loss(recon_gray, density)

            loss_aux = loss_gray + loss_high + loss_low + loss_density

            loss_bbox = reconstruction_loss(
                recon * bbox_mask,
                target * bbox_mask
            )

            loss = (
                LAMBDA_DIFF * loss_diff +
                LAMBDA_RECON * loss_recon +
                LAMBDA_AUX * loss_aux +
                LAMBDA_BBOX * loss_bbox
            )

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss_total += loss.item()

    train_loss = train_loss_total / len(train_loader)


    # -------------------------
    # VALIDATION (RECON MODE)
    # -------------------------
    controlnet.eval()
    fusion_module.eval()

    val_loss_total = 0
    real_feats = []
    fake_feats = []

    with torch.no_grad():

        for batch in val_loader:

            rgb = batch["rgb"].to(device)
            target = batch["xray_col"].to(device)

            gray = batch["gray"].to(device)
            high = batch["high"].to(device)
            low = batch["low"].to(device)
            density = batch["density"].to(device)

            bbox_mask = batch["bbox_mask"].to(device)
            colour = batch["colour"].to(device)

            # encode GT xray (reconstruction validation)
            latents = vae.encode(target).latent_dist.sample() * 0.18215

            noise = torch.randn_like(latents)

            timesteps = torch.randint(
                0,
                scheduler.config.num_train_timesteps,
                (latents.size(0),),
                device=device
            ).long()

            noisy_latents = scheduler.add_noise(latents, noise, timesteps)

            edges = compute_edge_map(rgb)

            control_input = torch.cat([
                edges,
                bbox_mask,
                gray,
                high,
                low,
                density
            ], dim=1)

            prompts = [
                f"airport x-ray image of a {m} {c}"
                for m, c in zip(batch["material"], batch["category"])
            ]

            text_inputs = tokenizer(
                prompts,
                padding="max_length",
                truncation=True,
                max_length=77,
                return_tensors="pt"
            ).to(device)

            text_emb = text_encoder(**text_inputs).last_hidden_state

            cls_token = text_emb[:,0,:]
            fused_cls = fusion_module(cls_token, colour)
            text_emb[:,0,:] = fused_cls

            encoder_hidden_states = text_emb

            down_samples, mid_sample = controlnet(
                noisy_latents,
                timesteps,
                encoder_hidden_states=encoder_hidden_states,
                controlnet_cond=control_input,
                return_dict=False
            )

            noise_pred = unet(
                noisy_latents,
                timesteps,
                encoder_hidden_states=encoder_hidden_states,
                down_block_additional_residuals=down_samples,
                mid_block_additional_residual=mid_sample
            ).sample

            alpha_t = alphas_cumprod[timesteps].view(-1,1,1,1)

            sqrt_alpha = torch.sqrt(alpha_t)
            sqrt_one_minus = torch.sqrt(1 - alpha_t)

            pred_x0 = (noisy_latents - sqrt_one_minus * noise_pred) / sqrt_alpha

            recon = vae.decode(pred_x0 / 0.18215).sample

            loss_recon = reconstruction_loss(recon, target)
            
            recon_gray = recon.mean(1, keepdim=True)
            
            loss_gray = reconstruction_loss(recon_gray, gray)
            loss_high = reconstruction_loss(recon_gray, high)
            loss_low = reconstruction_loss(recon_gray, low)
            loss_density = reconstruction_loss(recon_gray, density)
            
            loss_aux = loss_gray + loss_high + loss_low + loss_density
            
            loss_bbox = reconstruction_loss(
                recon * bbox_mask,
                target * bbox_mask
            )
            
            val_loss_batch = (
                LAMBDA_DIFF * loss_diff +
                LAMBDA_RECON * loss_recon +
                LAMBDA_AUX * loss_aux +
                LAMBDA_BBOX * loss_bbox
            )
            
            val_loss_total += val_loss_batch.item()
            
            recon_for_metrics = recon.clamp(0,1)
            
            real_feats.append(get_features(target))
            fake_feats.append(get_features(recon_for_metrics.detach()))

    val_loss = val_loss_total / len(val_loader)
    
    real_feats = torch.cat(real_feats)
    fake_feats = torch.cat(fake_feats)
    
    fid = compute_fid(real_feats, fake_feats)
    sid = compute_sid(real_feats, fake_feats)

    # -------------------------
    # BEST MODEL SAVE
    # -------------------------
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(
            controlnet.state_dict(),
            f"{SAVE_DIR}/best_controlnet.pt"
        )

    if epoch % 10 == 0:
        torch.save(
            controlnet.state_dict(),
            f"{SAVE_DIR}/controlnet_epoch_{epoch}.pt"
        )

    # -------------------------
    # LOGGING
    # -------------------------
    print("\n==============================")
    print(f"Epoch {epoch}")
    print(f"Train Loss : {train_loss:.4f}")
    print(f"Val Loss   : {val_loss:.4f}")
    print(f"FID        : {fid:.4f}")
    print(f"SID        : {sid:.4f}")
    print("GPU Memory (GB):", torch.cuda.memory_allocated()/1e9)
    print("==============================\n")

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 271/271 [01:29<00:00,  3.01it/s]



Epoch 0
Train Loss : 0.5917
Val Loss   : 0.5727
FID        : 40.3465
SID        : -0.0040
GPU Memory (GB): 10.523613184



100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 271/271 [01:29<00:00,  3.01it/s]



Epoch 1
Train Loss : 0.5942
Val Loss   : 0.5719
FID        : 37.8909
SID        : -0.0017
GPU Memory (GB): 10.520827904



100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 271/271 [01:30<00:00,  3.01it/s]


...

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 271/271 [01:27<00:00,  3.09it/s]



Epoch 33
Train Loss : 0.5877
Val Loss   : 0.5509
FID        : 33.9478
SID        : -0.0024
GPU Memory (GB): 10.50532864



100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 271/271 [01:27<00:00,  3.09it/s]



Epoch 34
Train Loss : 0.5846
Val Loss   : 0.5538
FID        : 32.4212
SID        : -0.0034
GPU Memory (GB): 10.506508288



 80%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▋                                   | 216/271 [01:10<00:17,  3.18it/s]